# ML-KEM encryption

# 0. References:

- https://www.pq-crystals.org/kyber/index.shtml
- https://csrc.nist.gov/pubs/fips/203/final

# 1. Introduction:

- ML-KEM (old name: Crystal-Kyber, standardized name FIPS 203) is an IND-CCA2-secure key encapsulation mechanism (KEM), whose security is based on the hardness of solving the learning-with-errors (LWE) problem over module lattices. A key-encapsulation mechanism (KEM) is a set of algorithms that can be used to establish a shared secret key between two parties communicating over a public channel.

- ML-KEM is one of the finalists in the NIST post-quantum cryptography project. ML-KEM is an approved alternative that is presently believed to be secure, even against adversaries in possession of a large-scale fault-tolerant quantum computer. The security of ML-KEM is based on the presumed hardness of the so-called Module Learning with Errors (MLWE) problem on module lattices.

- This standard specifies three parameter sets for ML-KEM that offer different trade-offs in security strength versus performance. All three parameter sets of ML-KEM are approved to protect sensitive, non-classified communication systems of the U.S. Federal Government. Specifically:

    + ML-KEM-512 aims at security roughly equivalent to AES-128,
    + ML-KEM-768 aims at security roughly equivalent to AES-192,
    + and ML-KEM-1024 aims at security roughly equivalent to AES-256.

- A worth-taking note is that, ML-KEM is "The Locksmith" - it securely creates a shared secret key between two parties over a public channel. To actually transfer data securely over the channel, you need something being "The Safe" like AES-256-GCM - it takes the shared secret key generated, use it as a password, and lock up your actual data (bulk text, files, videos, etc.).

# 2. Overview:

- A KEM consists of three algorithms and a collection of parameter sets. The three algorithms are:
    + A probabilistic key generation algorithm (denoted by ML-KEM.KeyGen)
    + A probabilistic ”encapsulation” algorithm (denoted by ML-KEM.Encaps)
    + A deterministic ”decapsulation” algorithm (denoted by ML-KEM.Decaps)
- How KEM works:
    + Alice begins by running KeyGen in order to generate a (public) encapsulation key and a (private) decapsulation key. 
    + Upon obtaining Alice’s encapsulation key, Bob runs the Encaps algorithm, which produces Bob’s copy $𝐾$ of the shared secret key along with an associated ciphertext. 
    + Bob sends the ciphertext to Alice.
    + Alice completes the process by running the Decaps algorithm using her decapsulation key and the ciphertext. This final step produces Alice’s copy $𝐾'$ of the shared secret key.
    + After completing this process, Alice and Bob would like to conclude that their outputs satisfy $𝐾' = 𝐾$ and that this value is a secure, random, shared secret key.

    However, these properties only hold if certain important conditions are satisfied, as discussed in SP 800-227.

- At a high level, the construction of the scheme ML-KEM proceeds in two steps. 
    + First, it construct a public-key encryption (K-PKE) scheme from the MLWE problem (MLWE problem is to recover $x$ from a set of random “noisy” linear equations in some secret variables $𝑥 \in \mathbb{R}_k^q$ - $𝑘$-fold Cartesian product of a certain polynomial ring $\mathbb{R}_q$.). This base scheme is not fully secure on its own against certain active attacks, thus shall not be used as a stand-alone cryptographic scheme.
    + Second, it applies a cryptographic technique known as a variant of the Fujisaki-Okamoto (FO) transform to convert the base K-PKE scheme into the fully secure (IND-CCA2-secure) KEM.

- A notable feature of ML-KEM's design is its use of the Number-Theoretic Transform (NTT), which allows the algorithm to perform extremely fast and efficient multiplication of polynomials without requiring excessive memory. The NTT converts a polynomial $𝑓 \in \mathbb{R}_q$ to an alternative representation as a vector $\hat{f}$ of linear polynimals.

# 3. Parameter Set:

- ML-KEM comes equipped with three parameter sets:
    + ML-KEM-512 (security category 1)
    + ML-KEM-768 (security category 3)
    + ML-KEM-1024 (security category 5)
    
    Regardless of whether usage of ML-KEM-512, ML-KEM-768, or ML-KEM-1024, the ML-KEM encapsulation and decapsulation algorithms always output a 256-bit shared secret key.

- The collection of parameter sets is used to select a trade-off between security and efficiency. Each parameter set in the collection is a list of specific (typically numerical) values ($𝑘, \eta_1, \eta_2, 𝑑_𝑢, 𝑑_𝑣$), one for each parameter required by the three algorithms. In addition to these five variable parameters, there are also two constants: $𝑛 = 256$ and $𝑞 = 3329$.
    + ML-KEM-512: (2,3,2,10,4)
    + ML-KEM-768: (3,2,2,10,4)
    + ML-KEM-1024: (4,2,2,11,5)

- When initially establishing cryptographic protections for data, the strongest possible parameter set should be used. This has a number of advantages, including reducing the likelihood of costly transitions to higher-security parameter sets in the future. 

    At the same time, it should be noted that some parameter sets might have adverse performance effects for the relevant application (e.g., the algorithm may be unacceptably slow, or objects such as keys or ciphertexts may be unacceptably large).

    NIST recommends using ML-KEM-768 as the default parameter set, as it provides a large security margin at a reasonable performance cost. Therefore, we willfollow, the same suit of biulding ML-KEM-768.

In [25]:
# Global constant of ML-KEM
n = 256
q = 3329

# Global constant of ML-KEM-768:
k = 3
eta_1 = 2
eta_2 = 2
d_u = 10
d_v = 4

# Some other global constant
zita = 17 # primitive n-th root of unity modulo q 

# 4. Theorem Base:

## 4.1. LWE, RLWE and MLWE problem:

## 4.2. Number-Theoretic Transform (NTT):

# 5. Auxiliary Algorithms:

In [18]:
import os
import secrets
import hashlib
import numpy

## 5.1. Cryptographic Functions:

In [ ]:
# A fresh string of random bytes must be generated for every such invocation. 
# These random bytes shall be generated using an approved RBG, as prescribed in SP 800-90A, SP 800-90B, and SP 800-90C
# Moreover, this RBG shall have a security strength of at least 128 bits for ML-KEM-512, at least 192 bits for ML-KEM-768, and at least 256 bits for ML-KEM-1024.

def RandomBitGeneration32() -> bytes:
    """Generate 32 random bytes using a cryptographically secure random number generator."""
    # For easy, we will temporarily use Python's secret library to generate RBG
    return secrets.token_bytes(32)

def HashH(s: bytes) -> bytes:
    """Take one variable-length input, using SHA3-256, and produce one 32-byte output"""
    return hashlib.sha3_256(s).digest()

def HashG(s: bytes) -> tuple[bytes, bytes]:
    """Take one variable-length input, using SHA3-512, and split into two 32-byte outputs"""
    res = hashlib.sha3_512(s).digest()
    return res[:32], res[32:64]

def PRF(eta: int, s: bytes, b: bytes) -> bytes: #Psuedo Randomize Function
    """Takes a parameter eta ∈ {2,3}, one 32-byte input, and one 1-byte input, using SHAKE256, produces one (64*eta)-byte output."""
    # If the 1-byte input is passed as an integer, convert it to bytes
    if isinstance(b, int):
        b = bytes([b])

    # Initialize SHAKE256 and digest the required number of bytes (64 * eta)
    return hashlib.shake_256(s+b).digest(64 * eta)

## 5.2. Conversion and Compression Algorithms:

In [ ]:
def BytesToBits(B):
    """Converting a byte array into a bit array."""
    pass

def ByteEncode(B, d):
    """Decodes a byte array into an array of d-bit integers for 1 ≤ d ≤ 12."""
    pass

def BitRev(r, n):
    """Bit reversal of a n-bit integer r"""
    pass


## 5.3. Sampling Algorithms:

In [21]:
def SampleNTT(B):
    """Takes a 32-byte seed and two indices as input and outputs a pseudorandom element of T_q."""
    pass

def SamplePolyCBD(B, eta):
    """Takes a seed as input and outputs a pseudorandom sample from the distribution D_eta(R_q)"""
    pass

## 5.4. The Number-Theoretic Transform:

In [ ]:
def NTT(f: numpy.ndarray) -> numpy.ndarray: #Number-Theoretic Transform:
    """Computes the NTT representation f_hat of ̂the given polynomial f ∈ R_q
    Input: the coefficients of the input polynomial (Z_q^256) | Output: the coefficients of the NTT of the input polynomial (Z_q^256)"""
    f_hat = f.copy() # We copy so we don't accidentally mutate the original polynomial
    i = 1

    length = 128; start = 0
    while (length >= 2):
        while (start < 256):
            zeta = (zita ** BitRev(i, 7)) % q # there will be a lookup table to avoid massive calculate
            i += 1

            j = start
            while (j < start + len): # this while loop done modulo q
                t = (zeta * f_hat[j + length]) % q
                f_hat[j + length] = (f_hat[j] - t) % q
                f_hat[j] = (f_hat[j] + t) % q

                j += 1
            start += start + 2*length
        length //= 2

    return f_hat

def BaseCaseMultiply(a_0, a_1, b_0, b_1, gamma):
    """Computes the product of two degree-one polynomials with respect to a quadratic modulus."""
    pass

def MutiplyNTTs(f_hat, g_hat):
    """Computes the product (in the ring $T_q$) of two NTT representations.
    Input: the coefficients of 2 NTT representations (in Z_q^256) | Output: the coefficients of the product of the inputs (Z_q^256)"""
    pass


# 6. ML-KEM Key Generaion:

## 6.1. KeyGen internal:

- The key generation algorithm K-PKE.KeyGen of K-PKE receives a seed as input and outputs an encryption key ek_PKE and a decryption key dk_PKE. 

- As is typically the case for public-key encryption, the encryption key can be made public, while the decryption key and the randomness
must remain private. 

- The matrix $\pmb{\hat{A}}$ generated in K-PKE.KeyGen can be stored, allows later operations to use $\pmb{\hat{A}}$ directly rather than re-expanding it from the public seed $\rho$.

- Informal description: 
    + The decryption key of K-PKE.KeyGen is a length-$k$ vector $\pmb{s}$ of elements of $\mathbb{R}_q$ (i.e., $\pmb{s} \in \mathbb{R}_q^k$). Roughly speaking, $\pmb{s}$ is a set of secret variables, while the encryption key is a collection of “noisy” linear equations ($\pmb{A}$,$\pmb{A}\pmb{s} + \pmb{\varepsilon}$) in the secret variables $\pmb{s}$. 
    + The rows of the matrix $\pmb{A}$ form the equation coefficients. This matrix is generated pseudorandomly using XOF with only a seed stored in the encryption key. The secret $\pmb{s}$ and the “noise” $\pmb{\varepsilon}$ are sampled from centered binomial distributions using randomness expanded from another seed $\sigma$ via pseudorandom function (PRF). 
    + Once $\pmb{A}$, $\pmb{s}$, and $\pmb{\varepsilon}$ are generated, the computation of the final part $\pmb{𝐭} = \pmb{𝐀}\pmb{𝐬} + \pmb{\varepsilon}$ of the encryption key takes place. The results are appropriately encoded into byte arrays and output.

In [ ]:
def KPKEKeygen(d: bytes) -> tuple[bytes, bytes]:
    """Uses randomness to generate an encryption key and a corresponding decryption key.
    Input: A randomness 32-byte string | Output: Encryp 384*k + 32 byte string,  Decryp 768*k + 96 byte string"""
    rho, sigma = HashG(d + bytes([k])) # expand 32+1 bytes to two pseudorandom 32-byte seeds

    # Generate k*k matrix \hat{A} ∈ (Z_q^256) 
    A_hat = numpy.empty((k, k), dtype=object)
    N = 0
    for i in range(k):
        for j in range(k):
            A_hat[i, j] = SampleNTT(rho + bytes([j, i])) #j and i are bytes 33 and 34 of the input

    # Generate s \in (Z_q^256)^k
    s = numpy.empty((1,k), dtype=object)
    for i in range(k):
        s[i] = SamplePolyCBD(PRF(sigma, N, eta_1), eta_1) # s[i] sampled from CBD
        N += 1

    # Generate e \in (Z_q^256)^k
    e = numpy.empty((1,k), dtype=object)
    for i in range(k):
        e[i] = SamplePolyCBD(PRF(sigma, N, eta_1), eta_1) # e[i] sampled from CBD
        N += 1

    # Run NTT
    s_hat = NTT(s)
    e_hat = NTT(e)
    
    t_hat = MutiplyNTTs(A_hat, s_hat) + e_hat # noisy linear system in NTT domain

    ek_pke = ByteEncode(t_hat, 12) + bytes([rho])
    dk_pke = ByteEncode(s_hat, 12)

    return ek_pke, dk_pke

- The algorithm ML-KEM.KeyGen_internal accepts two random seeds as input, and produces an encapsulation key and a decapsulation key.

In [16]:
def MLKEMKeyGen_internal(d: bytes, z: bytes) -> tuple[bytes, bytes]:
    """Uses randomness to generate an encapsulation key and a corresponding decapsulation key.
    Input: 2 randomness 32-byte string | Output: Encaps 384*k + 32 byte string,  Decaps 768*k + 96 byte string"""
    ek_pke, dk_pke = KPKEKeygen(d) # Run key generation for K-PKE

    #KEM encaps key is just the PKE encryption key
    ek = ek_pke
    
    #KEM decaps key includes PKE decryption key, the encapsulation key, a hash of the encapsulation key, and a random 32-byte value
    dk = dk_pke + ek + HashH(ek) + z

    return ek, dk

## 5.2. Main KeyGen algorithm:

- The key generation algorithm ML-KEM.KeyGen for ML-KEM accepts no input, generates randomness internally, and produces an encapsulation key and a decapsulation key.

- While the encapsulation key can be made public, the decapsulation key shall remain private.

In [22]:
def MLKEMKeyGen() -> tuple:
    """Generates an encapsulation key and a corresponding decapsulation key.
    No Input | Output: Encaps 384*k + 32 byte string,  Decaps 768*k + 96 byte string"""
    d = RandomBitGeneration32() # d is 32 random bytes
    z = RandomBitGeneration32() # z is 32 random bytes

    if (d == None) or (z == None):
        print("Error")
        return 0

    # Run internal key generation algorithm
    ek, dk = MLKEMKeyGen_internal(d,z)

    return ek,dk